In [ ]:
!pip install trimesh
!pip install triangle 
!pip install shapely

In [ ]:
import os
os.chdir("..")

import glob
import numpy as np 
import re 

import trimesh
import cv2

from skimage import io
from skimage import filters


from scipy.ndimage import rotate
from scipy import ndimage as ndi
%matplotlib qt

from numpy2stl.numpy2stl import numpy2stl, triangles_to_facets, writeSTL
from numpy2stl.numpy2stl import Solid

from geo2stl.geo2stl import *
from geo2stl import geo2stl as g2s
from geo2stl import sat2stl as s2s

from city2stl import dem2stl as d2s
import figure as figs 

def normalize(imx):

	l1,l2,l3= np.percentile(imx.ravel()[::10], [1,50,99])

	imx = imx - l2
	imx[imx<0] = imx[imx<0] / np.maximum(l1,l2-1)
	imx[imx>0] = imx[imx>0] / l3
	#imx = imx.clip(-5,5)
	return imx

def resize_geo_aspect(im, NSEW):
	lat = NSEW[[0,1]]
	lon = NSEW[[2,3]]

	lon_adj = lon * np.cos( np.deg2rad(lat).mean() )
	ratio = (np.diff(lon_adj ) / np.diff(lat))[0]
	shp = np.array(im.shape)[[1,0]]
	shp = np.array([1, 1/ratio]) * 1000
	sz = tuple((shp).astype(int))
	im = cv2.resize(im,sz)
		
	return im


def resize_max(im, max_size=1000):
	"""
	Resize an image to fit within a maximum size while maintaining aspect ratio.
	
	Parameters:
	- im: Input image as a 2D numpy array.
	- max_size: Maximum size for the longest dimnsion of the image.
	
	Returns:
	- Resized image as a 2D numpy array.
	"""
	height, width = im.shape
	scale = max_size / max(height, width)
	new_size = (int(width * scale), int(height * scale))
	resized_im = cv2.resize(im, new_size, interpolation=cv2.INTER_LINEAR)
	return resized_im

################### Write to file ##########################

def rescale(im, max_size=600, height=20, base=0.1, clip= None, smooth=None):
	
	im = resize_max(im, max_size=max_size)

	if smooth is not None:
		im = filters.median(im, np.ones((smooth,smooth)))
	
	
	if clip is not None:
		if len(clip)==1:
			clip = [clip, 100-clip]

		lo,hi = np.percentile(im.ravel(), clip)
		im = im.clip(lo,hi)	
	
	im  = im - im.min() + im.ptp()*base
	im = im / im.ptp()*height

	return im 


def subtract_water(dem, aquatic, height = 0.2, ocean_level=1):

	dem = dem.copy()
	im_a = aquatic.copy().astype(np.uint8)


	dem = resize_max(dem, max_size=1200)
	im_a = filters.median(im_a, np.ones((3,3)))
	im_a = cv2.resize(im_a, 
			(dem.shape[1], dem.shape[0]), 
			interpolation=cv2.INTER_LINEAR).astype(int)

	im_a = (im_a / 100)
	im_a = (im_a).clip(0,1)
	im_a[dem<0] = ocean_level
	dem = dem  - im_a*np.ptp(dem.ravel())*height

	return dem


In [2]:
import matplotlib.pyplot as plt

#### Get data from source and place in root
https://www.gebco.net/data-products/gridded-bathymetry-data/gebco-2020

""

In [3]:
Ocean_root = "C:/Users/eac84/Desktop/Desktop/Tasks/srtm_tifs/gebco_2020_geotiff/"
tile_files = glob.glob(Ocean_root+"*.tif")
out_dir = "C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Map_files/"

dem_root = "C:/Users/eac84/Desktop/Desktop/FILES"

# North America

## Wiss

In [ ]:
N =  40.085
E =  -75.182
S =  40.010
W =  -75.235

In [ ]:
bounds = (W, S, E, N)

dem = d2s.DEM(root=dem_root,geo_bounds =(N,S,E,W))
im = dem.data.copy()

filt = filters.gaussian(im,5)
im = im - filt*0.5
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))


lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)

im = rescale(im, max_size=600, height=20, base=0.1)
	



srtm_21_04


In [ ]:
name = "Wiss"
savefile(out_dir, name, im+1)

## Applelachans 

In [ ]:
(N, S, E, W) =  48, 33,  -64, -88


In [ ]:
target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()
im[im<0] = im[im<0]*0.1
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = rescale(im, max_size=600, height=25, base=0.1)

In [ ]:
name = "Appalachia"
savefile(out_dir, name, im)

## Penn

In [15]:
(N, S, E, W) =  42.8, 39.5,  -73.75, -80.


In [ ]:

target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = result.copy()

im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05

im = rescale(im, max_size=800, height=35, base=0.1,
         clip=None , smooth=[.1,99.95])

==== Stitching tiles without rasterio ====
Target bounding box: (42.8, 39.5, -73.75, -80.0)
✅ Finished stitching without rasterio.


NameError: name 'rescale' is not defined

In [ ]:
name = "Penn"
savefile(out_dir, name, im)

## Great Lakes

In [ ]:
(N, S, E, W) =  49.677, 40.465,  -74, -93.387
#(N, S, E, W) =  49.677, 41,  -70, -94
(N, S, E, W) =  49.5, 41,  -74, -94

In [ ]:

target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)


im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]
im = im * 0.1


im = rescale(im, max_size=800, height=25, base=0.1,
         clip=[.01,99.99], smooth=3)





In [ ]:
name = "GreatLakes_"
savefile(out_dir, name, im)

In [ ]:
import napari
print("opening - Napari")


tri = numpy2stl(im)
solid = Solid(tri)
vertices = solid.vertices.copy().astype(float)
faces = solid.faces


In [ ]:
mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
mesh.show()

In [ ]:

# Simplify mesh to 10% of original face count

mesh = trimesh.Trimesh(vertices=vertices, faces=faces)
simplified = mesh.simplify_quadric_decimation(0.9)
vertices = simplified.vertices.copy().astype(float)
faces = simplified.faces
#


In [ ]:
simplified.export(out_dir + "simplified.stl")

In [ ]:
v = napari.current_viewer()
if v is None: v = napari.Viewer()
v.layers.clear()

surface = (vertices,faces)
s = v.add_surface(surface)
s.wireframe.visible = True
s.wireframe.color = '#AAA'
v.dims.ndisplay = 3
s.shading = 'smooth' 
s.opacity = 0.8
v.axes.visible = True
v.axes.colored = True

## INDIAN RIVER

In [ ]:
N,S,E,W = 28,27,-80,-80.6

In [ ]:
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im[im>0] = im[im>0] + 10
im[im<-20] = -20



## Colorado

In [ ]:
N,S,E,W = [  41.0034002,   36.9925245, 
       	   -102.041585 , -109.0601879]


In [ ]:
target_bbox = np.array((N, S, E, W))
NSEW = np.array([N,S,E,W])

result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

x = 1500
im[im>x] = 0.3*(im[im>x]-x)+x

im = im * 0.02

im = resize_geo_aspect(im, NSEW)

filt = ndi.gaussian_filter(im, sigma=50)
im = im - filt*0.7

#im = filters.median(im, np.ones((3,3)))

im = rescale(im, max_size=800, height=50, base=0.1,
         clip=[.01,99.99], smooth=3)



In [ ]:
name = "Colorado1"
savefile(out_dir, name, im)

# Central America

## Caribbean

In [5]:
N,S,E,W = 31,6,-54,-102

In [8]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im[im>0] = im[im>0]+1000

#im3 = proj_map_height(im2, np.array([n,s,e,w]))
im = proj_map_geo_to_2D(im, target_bbox )

im = rescale(im, max_size=1000, height=30, base=0.1,
         clip=[.01,99.99], smooth=None)


NameError: name 'tile_files' is not defined

In [ ]:
tri = np3D.numpy2stl(im4)
vertices, faces = np3D.vertices_to_index(tri)
mesh = trimesh.Trimesh(vertices,faces)
mesh.show()

In [ ]:
facets = np3D.triangles_to_facets(tri)
np3D.writeSTL(facets, "Ocean_flat.stl")

## Puerto Rico

In [ ]:
N,S,E,W  = 18.75, 17.65, -65.15, -67.35

In [ ]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

im[im<0] = im[im<0]*0.2
im[im>0] = im[im>0] + 100

im_x = im - im.min() + (im.ptp()*0.1)

scale = 2
sz = tuple((np.array(im_x.shape)[[1,0]]*scale).astype(int))
im_x = im_x * scale
im_x = cv2.resize(im_x,sz)
im_x = im_x * 0.02


In [ ]:
name = "PuertoRico"
savefile(out_dir, name, im_x)

## Keys

In [ ]:
(N, S, E, W) = 25.9,24.3, -80,-82.0

In [ ]:
target_bbox = (N, S, E, W)

result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]

im[im<-100] = -100
im[im<0] = im[im<0]*0.1
im[im>0] = im[im>0] + 10

#im[im<0] = im[im<0]-200

scale = 2
im_x = im.copy()
sz = tuple((np.array(im_x.shape)[[1,0]]*scale).astype(int))
im_x = im_x * scale
im_x = cv2.resize(im_x,sz)
im_x = im_x * 0.02

DEM = im_x
rotation = 55 - 90

DEM = rotate(DEM,rotation, reshape=True, cval=-10)

DEM = DEM[480:820,:]
DEM[DEM==-10] = DEM[DEM>-10].min()




## Bahamas

In [ ]:
N,S,E,W = 35,10,-60,-88


In [ ]:

target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()

im = proj_map_geo_to_2D(im, target_bbox , clip_out=False)
im[np.isnan(im)] = -10000

DEM = im.copy()
lo,hi = np.percentile(DEM.ravel(), [.01,99.99])
DEM = DEM.clip(lo,hi)

#DEM = DEM[::10,::10]
rotation = 38

DEM = rotate(DEM,rotation, reshape=True, cval=-0)

x1,x2,y1,y2 = 3800,6100, 2200,6700

DEM2 = DEM[x1:x2, y1:y2].copy()

lo,hi = np.percentile(DEM.ravel(), [.01,99.99])
DEM = DEM.clip(lo,hi)

DEM2[DEM2>0] = DEM2[DEM2>0]+np.ptp(DEM2)*0.1
DEM2 = resize_max(DEM2, max_size=1000)
DEM2 = DEM2 - DEM2.min() + DEM2.ptp()*0.1
DEM2 = DEM2 / DEM2.ptp() * 25

plt.close("all")	
plt.figure()
plt.imshow(DEM,cmap="jet")
plt.plot([y1,y1,y2,y2,y1],[x1,x2,x2,x1,x1], color="white", linewidth=3)
plt.grid(True)

plt.figure()
plt.imshow(DEM2,cmap="jet")
plt.grid(True)


plt.figure()
_ = plt.hist(DEM2.ravel(), bins=100)

In [ ]:
name = "Antillas_38deg"
savefile(out_dir, name, DEM2)

## Mexico

In [25]:
N,S,E,W = 37,11,-75,-122

target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

==== Stitching tiles without rasterio ====
Target bounding box: [  37   11  -75 -122]
✅ Finished stitching without rasterio.


In [27]:
im = result.copy()

im = resize_max(im, max_size=1000)
im = proj_map_geo_to_2D(im, target_bbox )


im[im<0] = im[im<0] * 0.05
im[im<0] = im[im<0] - im.ptp()*0.1
im = im / im.ptp()*30
im  = im - im.min() + im.ptp()*0.1

figs.plot_data(im)


# South America

## COLOMBIA

In [ ]:
from strm2stl.osm2stl import get_boundries_osmnx
from strm2stl.create import coor2im
from strm2stl.dem2stl import embed_lines
from skimage import morphology
from skimage.draw import line_aa, polygon2mask

In [ ]:
def get_border(nsew, country, shape):

	bounds,bbox = get_boundries_osmnx(nsew, country)
	boundry = bounds[-2]
		
	
	coorlims = (nsew[np.array([0,1])], nsew[np.array([2,3])])
	imlims = ((0,shape[1]), (shape[0], 0))
	r,c = coor2im(coorlims, imlims, boundry.pts.T)

	im_lines = np.zeros(shape, dtype=np.uint8)

	for i in range(len(r)-1):
		rr, cc, val = line_aa(int(round(r[i])), int(round(c[i])), int(round(r[i+1])), int(round(c[i+1])))
		im_lines[rr, cc] = val*255
			
	im_lines = morphology.binary_dilation(im_lines, morphology.disk(2))
		
	return im_lines
  




In [ ]:
N,S,E,W = 14,-6,-64,-84
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

In [ ]:
x1 = 300 
x3 = 2000
im_ = im.copy()
im_[im_>x1] = im_[im_>x1] + (x3-x1)
bx = (im_<=x1) & (im_>0)
im_[bx] = im_[bx] * (x3/x1)

im_[im_<0] = im_[im_<0]*0.5
im_[im_>0] = im_[im_>0] + 10

#############
scale = 0.5
sz = tuple((np.array(im_.shape)[[1,0]]*scale).astype(int))
im_ = im_ * scale
im_ = cv2.resize(im_,sz)
im_ = im_ * 0.01

###############
sealine = im_>0
sealine = morphology.binary_dilation(sealine, morphology.disk(2)) & ~sealine
im_ = im_ - (sealine* im_.ptp()*0.01)

nsew = np.array([N,S,E,W])


In [ ]:
im_lines = get_border(nsew, "Colombia", im_.shape)
im_lines[im_<0] = 0 
im_ = im_ - (im_lines* im_.ptp()*0.01)
im_ = im_ - np.min(im_) + (im_.ptp()*.01)

In [ ]:

name = "Colombia"
savefile(out_dir, name, im_ )

## Medellin

## Brazil

In [28]:
im_all = io.imread(tile_files[1])
plt.imshow(im_all[::10,::10])

In [29]:
plt.close("all")

imx = im_all[5500:5800,10200:10700]*1.
imx = normalize(imx)
plt.figure()
plt.imshow(imx, cmap="jet")

imx = im_all[5420:5550,11150:11350]*1.
imx = normalize(imx)
plt.figure()
plt.imshow(imx, cmap="jet")


imx = im_all[5200:5800,10200:11500]*1.
imx[imx>0] = imx[imx>0]+100
imx = imx.clip(-100)
imx = normalize(imx)

plt.figure()
plt.imshow(imx, cmap="jet")

In [30]:
plt.close("all")

imx = im_all[4000:7000:10,7500:10500:10]*1.
imx = imx.clip(-10)
imx[imx>0] = imx[imx>0]+200
imx = normalize(imx)
plt.figure()	
plt.imshow(imx, cmap="jet")

imx = im_all[5800:6500,8000:9000:]*1.
imx = normalize(imx)

plt.figure()	
plt.imshow(imx, cmap="jet")

imx = im_all[5900:6200,8300:8800:]*1.
imx = normalize(imx)

plt.figure()	
plt.imshow(imx, cmap="jet")
plt.figure()
plt.hist(imx.ravel())

(array([  234.,   562.,  8010., 39075., 42921., 23939., 13317., 12008.,
         7669.,  2265.]),
 array([-0.63669065, -0.50099361, -0.36529657, -0.22959954, -0.0939025 ,
         0.04179454,  0.17749157,  0.31318861,  0.44888565,  0.58458268,
         0.72027972]),
 <BarContainer object of 10 artists>)

## Amazon

In [7]:
s2s.initialize_earth_engine()
N,S,E,W = 14,-20,-45,-85


In [10]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)
img = s2s.get_aquatic_regions(N,S,E,W, dataset="jrc", scale=2000)

im = result.copy()
im = subtract_water(im, img, height = 0.2, ocean_level=1)

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))
im = rescale(im, max_size=600, height=30, base=0.2, clip= [.01,99.99], smooth=3)


figs.plot_data(im)

plt.figure()
img2 = resize_max(img*1., max_size=600)
plt.imshow(img2, cmap="jet")

name = "Amazon"
#savefile(out_dir, name, im)

==== Stitching tiles without rasterio ====
Target bounding box: (14, -20, -45, -85)
✅ Finished stitching without rasterio.
(1894, 2228)


# Europe

## norway

In [ ]:
N=72.07
S=57.03
E= 33.28 
W= -3

In [ ]:
target_bbox = (N, S, E, W)

result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im[im<0] = im[im<0]*0.1

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = resize_max(im, max_size=800)

## North Sea

In [ ]:
N=67
S=53
E= 38
W= -2



In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

if 1:
	im[im<0] = np.sqrt(np.abs(im[im<0])) * np.sign(im[im<0])
	im[im<0] = im[im<0] / (np.ptp(im[im<0])) * result[im<0].ptp()*0.5
	
if 0:

	im[im>0] = np.sqrt(np.abs(im[im>0])) * np.sign(im[im>0])
	im[im>0] = im[im>0] / (np.ptp(im[im>0])) * result[im>0].ptp()

im[im>0] = im[im>0] + np.ptp(im.ravel())*0.05

im = proj_map_geo_to_2D(im,np.array((N, S, E, W)), clip_out=True)
im = im.clip(*np.percentile(im.ravel(), [.5,99.9]))
im = resize_max(im, max_size=800)
im = ndi.median_filter(im, size=3)

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 25



In [ ]:
name = "NorthSea"
savefile(out_dir, name, im)

## IBERIA

In [ ]:
N,S,E,W = 44.4,34.8,5.3,-11

In [ ]:
target_bbox = np.array((N, S, E, W))
result = stitch_tiles_no_rasterio(tile_files, target_bbox)
im = result.copy()

im[im<0] = im[im<0]*0.25
im[im>0] = im[im>0]+100

im_r = im.copy()
im_r[im_r<0]=0
im_r = (ndi.gaussian_filter(im_r,10) - im_r).clip(0)
im = im - 2*im_r


im2 = proj_map_geo_to_2D(im, np.array(nsew))
im2 = im2 - np.min(im2) + 1


scale = 0.3
sz = tuple((np.array(im2.shape)[[1,0]]*scale).astype(int))
im2 = im2 * scale
im2 = cv2.resize(im2,sz)
im2 = im2 * 0.02


In [ ]:
name = "Ibera"
savefile(out_dir, name, im2 )


## Mediterrian 

In [35]:
N, S, W, E =  48.5, 28.5, -15, 45.2


In [43]:
target_bbox = (N, S, E, W)
result = stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im = proj_map_geo_to_2D(im,np.array((N, S, E, W)), True)
im = resize_max(im, max_size=800)

lo,hi = np.percentile(im.ravel(), [5,99.99])
im = im.clip(lo,hi)

im[im<0] = im[im<0]*0.8
im[im<0] = im[im<0]-im.ptp()*0.05

im = im / im.ptp()*30
im  = im - im.min() + im.ptp()*0.05


==== Stitching tiles without rasterio ====
Target bounding box: (48.5, 28.5, 45.2, -15)
✅ Finished stitching without rasterio.


In [44]:
figs.plot_data(im)

In [ ]:
view3D_napari(im)

In [ ]:
name = "Mediterranean"
savefile(out_dir, name, im)

# AFRICA

## Pan Africa

In [ ]:
im_4 = io.imread(strm_fns[2])
im_1 = io.imread(strm_fns[5])
im_2 = io.imread(strm_fns[6])
im_3 = io.imread(strm_fns[1])

NameError: name 'strm_fns' is not defined

In [ ]:
im_all = np.vstack([np.hstack([im_1,im_2]),np.hstack([im_3,im_4])])
index = [11000,31500,15500,36000]
im_0 = im_all[index[0]:index[1],index[2]:index[3]] * 1.0


In [ ]:
im_x = im_0
sz = tuple((np.array(im_x.shape)[[1,0]]*0.02).astype(int))
im_x = im_x * 0.1
im_x = cv2.resize(im_x,sz)*1.0

sz = tuple((np.array(im_x.shape)[[1,0]]*2).astype(int))
im_x = cv2.resize(im_x,sz)


im_x[im_x<0] = im_x[im_x<0]*.1
im_x[im_x<0] = im_x[im_x<0]*.2

x = 200
im_x[im_x>x] = 0.2*(im_x[im_x>x]-x)+x

x = 100
im_x[im_x>x] = 0.5*(im_x[im_x>x]-x)+x

x = 50
im_x[im_x>x] = 0.5*(im_x[im_x>x]-x)+x

x = -10
im_x[im_x<x] = 5*(im_x[im_x<x]-x)+x



im_x[im_x<0] = im_x[im_x<0] - 0.05*(im_x.max()-im_x.min())

n1,s1,e1,w1 = mat2coor( [0, 180, -90,90], [43200, 43200], index)
s1 = s1 - 180

nsew1 = (n1,s1,e1,w1)

im_x2 = proj_map_geo_to_2D(im_x,np.array(nsew1))

im_x = im_x - im_x.min() + 10

In [ ]:

name = "Africa"
savefile(out_dir, name, im_x)

# Asia 

## Bengal

In [ ]:
s2s.initialize_earth_engine()

In [ ]:
N,S,E,W =  29, 20 ,95, 84

==== Stitching tiles without rasterio ====
Target bounding box: (29, 20, 95, 84)
✅ Finished stitching without rasterio.
(1003, 1226)


In [ ]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)
img = s2s.get_aquatic_regions(N,S,E,W, dataset="jrc")

im = result.copy()
img2 = img.copy().astype(np.uint8)

im = resize_max(im, max_size=1200)
img2 = filters.median(img2, np.ones((3,3)))
img2 = cv2.resize(img2, (im.shape[1], im.shape[0]), interpolation=cv2.INTER_LINEAR).astype(int)
img2[im<0] = 200
img2[im>500] = 0
img2 = img2 / 100
img2 = (img2).clip(0,1)


x = 1
xmax = im.max()
im[im>x] = ((im[im>x]/im.max())**0.7 )*im.max()


im = im  - img2*np.ptp(im.ravel())*0.1

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = resize_max(im, max_size=600)
im = filters.median(im, np.ones((3,3)))
im = im.clip(*np.percentile(im.ravel(), [.01,99.99]))
im  = im - im.min() + im.ptp()*0.2
im = im / im.ptp() * 30




In [ ]:
name = "Bengal"
savefile(out_dir, name, im)

Saving Image
File already exists, saving as C:/Users/eac84/OneDrive/Documents/Projects/3D Maps/Map_files//Bengal/Bengal_1.npy
Creating top...
Creating walls...
Creating bottom...
Simplifying Surfaces...
Saving STL


## Persia

In [ ]:
N,S,E,W = 41,  24 ,65, 41


In [ ]:

target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))


im[im<0] = im[im<0]*0.2
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05
im  = im - im.min() + im.ptp()*0.1

im = resize_max(im, max_size=900)
im = filters.median(im, np.ones((3,3)))
lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)




In [ ]:
name = "Persia"
savefile(out_dir, name, im)

## Japans

In [31]:
N,S,E,W = 55.066, 28.75, 153.69, 118
N,S,E,W = 48, 30, 150, 122


In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()

im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))

im = im[:,~np.any(np.isnan(im),axis=0)]

im[im<0] = im[im<0]*0.2
im[im<0] = im[im<0]- np.ptp(im.ravel())*0.05
im[im<0] = im[im<0].clip(-4000)

im = resize_max(im, max_size=800)
im = filters.median(im, np.ones((3,3)))

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 30

==== Stitching tiles without rasterio ====
Target bounding box: (48, 30, 150, 122)
✅ Finished stitching without rasterio.


In [34]:
figs.plot_data(im)

In [33]:
name = "Japans"
savefile(out_dir, name, im)

NameError: name 'savefile' is not defined

## South China Sea

In [ ]:
N,S,E,W = 25.938, -11.716, 133.769, 91.58
N,S,E,W = 25.938, -11.716, 136, 88
N,S,E,W = 29, -14, 140 , 86


In [ ]:
target_bbox = (N, S, E, W)
result = g2s.stitch_tiles_no_rasterio(tile_files, target_bbox)

im = result.copy()
im = g2s.proj_map_geo_to_2D(im,np.array((N, S, E, W)))
im = resize_max(im, max_size=800)

lo,hi = np.percentile(im.ravel(), [.1,99.9])
im = im.clip(lo,hi)

im = filters.median(im, np.ones((3,3)))

im[im<0] = im[im<0]*0.8
im[im<0] = im[im<0] - im.ptp()*0.05

im  = im - im.min() + im.ptp()*0.1
im = im / im.ptp() * 30


In [ ]:

name = "SouthChinaSea"
savefile(out_dir, name, im)